# Generating samples and detecting anomalies with GMMs

The goal of the lab is to build a generative model of images with a GMM model. We will:
- Create a GMM class with methods implementing the E-step, M-step, and training loop.
- Implement methods allowing to compute the full likelihood (to check convergence) and sample images.
- We will use the model to detect *anomalies*, which will be numbers we left aside initially.

We will work with the ```digits``` datasets for simplicity - but you can apply it to MNIST easily. It is composed of ```8 x 8``` images with labels being digits (*0 to 9*).

### Loading data

In [1]:
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np

np.random.seed(0)

In [2]:
from sklearn.datasets import load_digits

digits = load_digits()
X = digits.data / 16.0       # shape (1797, 64) which we normalize into pixel values in [0, 1]
y = digits.target      # shape (1797,)

print(X.shape, y.shape)

(1797, 64) (1797,)


### Loading a subset of data

We create a small function to select a subset of the data according to the label:
<div class='alert alert-block alert-info'>
            Code:</div>

In [4]:
def load_digits_subset(digits, X, y):
    X_out = []
    y_out = []
    # Keep data with labels in digits
    for i in range(len(X)):
        if y[i] in digits:
            X_out.append(X[i])
            y_out.append(y[i])
    return np.array(X_out), np.array(y_out)
    
# Digits used for training 
train_digits = [0,1,2,3,4]
X_train, y_train = load_digits_subset(train_digits, X, y)

print("Training data shape:", X_train.shape)

Training data shape: (901, 64)


### Creating the GMM model

We create a class with the necessary hyperparameters:
- The number of clusters *K*
- The number of features *p*

And the necessary parameters:
- The mixture distribution $\pi$
- The means $\mu_k$
- The covariance matrix $\Sigma_k$. **To simplify the code, we limite our covariance matrix to be diagonal** and hence only need a scalar $\sigma_k$ for each cluster.


Use the ```log_gaussian_diag``` function defined in the next cell !
<div class='alert alert-block alert-info'>
            Code:</div>

In [30]:
class DiagonalGMM:
    def __init__(self, n_components, n_features):
        # Hyperparameters
        self.K = n_components
        self.p = n_features
        self.alpha = 1e-8 # To avoid underflow and probabilities rounded down to 0
        # Parameters
        self.pi = np.random.random(self.K) # Initialize to random distribution
        self.mu = np.random.normal(0, 1,size=self.p) # Initialize randomly using univariate gaussian
        self.sigma = np.eye(self.p, self.p) #Diagonal covariance matrix - initialize at one

    def e_step(self, X):
        n = X.shape[0]
        log_r = np.zeros((n, self.K))
        # Compute the log-posterior probabilities of the latent variable = the responsabilities
        for k in range(self.K):
            log_r[:,k]=np.log(self.pi[k])*log_gaussian_diag(X, self.mu[k], self.sigma[k])
        # Normalize them and obtain probabilities
        r = np.exp(log_r)
        L=np.sum(r,axis=1)
        r=r/L
        return r
    
    def m_step(self, X, r):
        # Updating pi 
        self.pi=(1/len(X))*np.sum(r, axis=1)
        # Updating mu 
        self.mu = np.array([np.sum([r[i,k]*X[i] for i in range(len(X))]) / np.sum(r, axis=1) for k in range(self.K)])
        # Updating sigma
        self.sigma=[np.sum([r[i,k]*(X[i]-self.mu[k])@(X[i]-self.mu[k]).T for i in range(len(X))])for k in range(self.K)]/ np.sum(r, axis=1)

    def log_likelihood(self, X):
        n = X.shape[0]
        log_prob = np.zeros((n, self.K))
        # Apply the formula
        # Easier to do iterating over clusters
        for k in range(self.K):
            log_prob[:,k]=np.log(self.pi[k])+log_gaussian_diag(X, self.mu[k], self.sigma[k])
        log_prob=np.mean(np.log(np.sum(np.exp(log_prob), axis=1)))
        return log_prob

    def fit(self, X, n_iters=20):
        for it in range(n_iters):
            print(f"mu {it}: {self.mu.shape}")
            # Apply e-step, m-step, and compute and return the likelihood:
            r = self.e_step(X)
            print(f"mu {it} estep: {self.mu.shape}")
            self.m_step(X, r)
            print(f"mu {it} mstep: {self.mu.shape}")
        likelihood = np.exp(self.log_likelihood(X))
        return likelihood

    def sample(self, n_samples):
        # Sample a vector of n_samples latent variable giving cluster id (from pi)
        # The sample is mu + a sample from a N(0,1) gaussian * sigma
        pass

We also need to implement the log-density computation for each distribution in order to compute the E-step. We can also use it to keep track of the likelihood during training:

<div class='alert alert-block alert-info'>
            Code:</div>

In [24]:
def log_gaussian_diag(X, mu, sigma):
    # Apply the formula of gaussian density, simplified since the covariance is diagonal
    sigma_inv = np.diag([1/sigma[i] for i in range(len(sigma))])
    det_sigma = 1
    for i in range(len(sigma)):
        det_sigma*=sigma[i]
    print(sigma_inv.shape)
    print(mu.shape)
    print(X.shape)
    return -np.log((np.sqrt(2*np.pi))**len(X[0]) * np.sqrt(det_sigma))+(1/2)*(X-mu).T @ sigma_inv @ (X-mu)

### Training the model

We can now train the model. We should use more clusters than needed: **why ?** 

Generally, **what would be the biggest issue with our training scheme** and what should we do to reduce it ? 

In [31]:
K = 8
D = X_train.shape[1]
# Create the model
gmm = DiagonalGMM(n_components=K, n_features=D)
# Train the model
gmm.fit(X_train, n_iters=20)

mu 0: (64,)
(64, 64)
()
(901, 64)


/tmp/ipykernel_1463/367952805.py:3: RuntimeWarning: divide by zero encountered in scalar divide
  sigma_inv = np.diag([1/sigma[i] for i in range(len(sigma))])
/tmp/ipykernel_1463/367952805.py:10: RuntimeWarning: divide by zero encountered in log
  return -np.log((np.sqrt(2*np.pi))**len(X[0]) * np.sqrt(det_sigma))+(1/2)*(X-mu).T @ sigma_inv @ (X-mu)


ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 64 is different from 901)

We can visualize the means $\mu_k$ learned by the GMM. However, we will see that they are **blurry** !
- Why are they blurry ?
- When setting up our model, what assumption did we make about the pixels ?
<div class='alert alert-block alert-info'>
            Code:</div>

In [ ]:
plt.figure(figsize=(20,4))
for k in range(K):
    plt.subplot(1, K, k+1)
    plt.imshow(..., cmap='gray')
    plt.axis('off')
plt.suptitle("Learned GMM means")
plt.show()

We can now generate a few samples by adding the appropriate method to the class. The process is easy:
- Pick a cluster $k$,
- Add the cluster mean $\mu_k$ to the variance times a randow draw $x \sim \mathcal{N}(0,1)$: $ \mu_k + \sigma_k \times x$

However, the results are not great. **What can we conclude about the likelihood of our model ?** What is missing ? 
<div class='alert alert-block alert-info'>
            Code:</div>

In [ ]:
# Get samples from the appropriate method. Also get the cluster id from which they were samples
...
plt.figure(figsize=(8,8))
for i in range(16):
    plt.subplot(4,4,i+1)
    plt.imshow(..., cmap='gray')
    plt.title(f"Cluster {...}", fontsize=8)
    plt.axis('off')
plt.suptitle("Generated samples")
plt.show()

### Detecting anomalies

We will now use the remaining labels to check if our model is able to detect them as *anomalies*. 

To do this, we need to compute their **likelihood according to our model**. Why should they have a lower likelihood than the other ? 

In [ ]:
left_out_digits = [5,6,7,8,9]
X_ood, y_ood = load_digits_subset(left_out_digits, X, y)

In [ ]:
# Compute the log-likelihoods
l_l_train = ...
l_l_ood = ...

n_bins = 50
bins = np.linspace(- ll_train.max(), ll_train.max(), n_bins + 1)

plt.hist(ll_train, bins=bins, alpha=0.7, label="In-distribution")
plt.hist(ll_ood, bins=bins, alpha=0.7, label="Out-of-distribution")
plt.legend()
plt.xlim(- ll_train.max(), ll_train.max())
plt.title("Log-likelihood for anomaly detection")
plt.show()